In [67]:
import pandas as pd
import numpy as np
from pathlib import Path
import unicodedata
import re


# ============================================================
# 1. KLASÖR AYARLARI
# ============================================================

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data" / "raw").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. YARDIMCI FONKSİYONLAR
# ============================================================

def clean_column_name(column_name):
    """
    Türkçe karakterleri ve özel karakterleri temizleyerek
    standart snake_case kolon isimleri oluşturur.
    """
    turkish_map = str.maketrans({
        "ç": "c", "Ç": "C",
        "ğ": "g", "Ğ": "G",
        "ı": "i", "İ": "I",
        "ö": "o", "Ö": "O",
        "ş": "s", "Ş": "S",
        "ü": "u", "Ü": "U"
    })

    text = str(column_name).translate(turkish_map)
    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", "ignore").decode("ascii")
    text = re.sub(r"[^a-zA-Z0-9]+", "_", text)
    text = re.sub(r"_+", "_", text)

    return text.strip("_").lower()


def clean_dataframe_columns(df):
    """
    Kolon isimlerini temizler ve metin alanlarındaki boşlukları giderir.
    """
    df = df.copy()

    new_columns = [clean_column_name(col) for col in df.columns]

    if len(new_columns) != len(set(new_columns)):
        raise ValueError("Kolon isimleri temizlendikten sonra duplicate kolon oluştu.")

    df.columns = new_columns

    for column in df.columns:
        if (
            pd.api.types.is_object_dtype(df[column])
            or pd.api.types.is_string_dtype(df[column])
        ):
            df[column] = df[column].map(
                lambda value: (
                    value.replace("\xa0", " ").strip()
                    if isinstance(value, str)
                    else value
                )
            )

            df[column] = df[column].replace("", pd.NA)

    return df


def normalize_status(series):
    """
    Türkçe İ / ı karakterlerinden kaynaklanan status problemlerini çözer.
    PyArrow uyumlu şekilde çalışır.
    """

    def normalize_one(value):
        if pd.isna(value):
            return pd.NA

        value = str(value)
        value = value.replace("\xa0", " ").strip()

        # Türkçe karakterleri parçala
        value = unicodedata.normalize("NFKD", value)

        # Aksan/combining karakterlerini kaldır
        value = "".join(
            character
            for character in value
            if not unicodedata.combining(character)
        )

        return value.casefold()

    return series.map(normalize_one).astype("string")


def join_unique_values(series):
    """
    Aynı sipariş içindeki farklı iade nedenlerini tek hücrede birleştirir.
    """
    values = []

    for value in series:
        if pd.notna(value):
            value = str(value).strip()

            if value and value not in values:
                values.append(value)

    if not values:
        return pd.NA

    return " | ".join(values)


def add_date_features(df, date_column, prefix):
    """
    Belirtilen tarih kolonundan yıl, ay, gün, hafta ve hafta günü feature'ları üretir.
    """
    if date_column not in df.columns:
        return df

    date_series = df[date_column]

    df[f"{prefix}_year"] = date_series.dt.year.astype("Int64")
    df[f"{prefix}_month"] = date_series.dt.month.astype("Int64")
    df[f"{prefix}_day"] = date_series.dt.day.astype("Int64")
    df[f"{prefix}_day_of_week"] = date_series.dt.dayofweek.astype("Int64")
    df[f"{prefix}_week_of_year"] = date_series.dt.isocalendar().week.astype("Int64")
    df[f"{prefix}_quarter"] = date_series.dt.quarter.astype("Int64")
    df[f"{prefix}_is_weekend"] = (
        date_series.dt.dayofweek.isin([5, 6]).astype("int8")
    )

    return df


# ============================================================
# 3. HAM VERİLERİ OKUMA
# ============================================================

df_teslim = pd.read_excel(
    RAW_DIR / "dogo_teslim_edilenler.xlsx"
)

df_iptal = pd.read_excel(
    RAW_DIR / "dogo_iptal.xlsx"
)

df_iade = pd.read_excel(
    RAW_DIR / "dogo_iade.xlsx"
)

df_iade_aciklamali = pd.read_excel(
    RAW_DIR / "dogo_iade_aciklamali.xlsx"
)


# Kolon isimlerini temizleme
df_teslim = clean_dataframe_columns(df_teslim)
df_iptal = clean_dataframe_columns(df_iptal)
df_iade = clean_dataframe_columns(df_iade)
df_iade_aciklamali = clean_dataframe_columns(df_iade_aciklamali)


# Kaynak bilgisi ekleme
df_teslim["source_dataset"] = "teslim"
df_iptal["source_dataset"] = "iptal"
df_iade["source_dataset"] = "iade"


# Sipariş numaralarını normalize etme
for dataframe in [
    df_teslim,
    df_iptal,
    df_iade,
    df_iade_aciklamali
]:
    dataframe["siparis_no"] = (
        dataframe["siparis_no"]
        .astype("string")
        .str.replace("\xa0", " ", regex=False)
        .str.strip()
    )


# ============================================================
# 4. İADE AÇIKLAMALI DOSYASINI SİPARİŞ SEVİYESİNE AGGREGATE ETME
# ============================================================

df_iade_aciklamali["talep_tarihi"] = pd.to_datetime(
    df_iade_aciklamali["talep_tarihi"],
    errors="coerce"
)

for column in [
    "siparis_edilen_miktar",
    "iade_edilecek_miktar"
]:
    if column in df_iade_aciklamali.columns:
        df_iade_aciklamali[column] = pd.to_numeric(
            df_iade_aciklamali[column],
            errors="coerce"
        )


return_summary = (
    df_iade_aciklamali
    .groupby("siparis_no", as_index=False)
    .agg(
        return_line_count=("siparis_no", "size"),

        return_reason_row_count=(
            "iade_nedeni",
            lambda series: series.notna().sum()
        ),

        return_reason=(
            "iade_nedeni",
            join_unique_values
        ),

        return_request_date=(
            "talep_tarihi",
            "min"
        ),

        return_qty_ordered=(
            "siparis_edilen_miktar",
            lambda series: series.sum(min_count=1)
        ),

        return_qty_requested=(
            "iade_edilecek_miktar",
            lambda series: series.sum(min_count=1)
        )
    )
)


# ============================================================
# 5. TESLİM + İPTAL + İADE VERİLERİNİ BİRLEŞTİRME
# ============================================================

df_core = pd.concat(
    [
        df_teslim,
        df_iptal,
        df_iade
    ],
    ignore_index=True
)


# Her siparişin tek satır olması bekleniyor
duplicate_order_count = df_core["siparis_no"].duplicated().sum()

if duplicate_order_count > 0:
    raise ValueError(
        f"Core veride {duplicate_order_count} duplicate sipariş bulundu."
    )


# İade özeti ile birleştirme
df_core = df_core.merge(
    return_summary,
    on="siparis_no",
    how="left",
    validate="one_to_one"
)


# Ana dosyalarda bulunmayan iade kayıtlarını raporlamak için
core_order_numbers = set(df_core["siparis_no"].dropna())

unmatched_return_detail = df_iade_aciklamali[
    ~df_iade_aciklamali["siparis_no"].isin(core_order_numbers)
].copy()


# ============================================================
# 6. SAYISAL VE TARİH KOLONLARINI DÜZENLEME
# ============================================================

date_columns = [
    "tarih",
    "fatura_tarihi",
    "return_request_date"
]

for column in date_columns:
    if column in df_core.columns:
        df_core[column] = pd.to_datetime(
            df_core[column],
            errors="coerce"
        )


numeric_columns = [
    "id",
    "doviz_tutar",
    "tutar",
    "kdv",
    "kargo_toplami",
    "hizmet_bedeli",
    "gecen_sure_dk",
    "kur_fiyati",
    "return_line_count",
    "return_reason_row_count",
    "return_qty_ordered",
    "return_qty_requested"
]

for column in numeric_columns:
    if column in df_core.columns:
        df_core[column] = pd.to_numeric(
            df_core[column],
            errors="coerce"
        )


# ============================================================
# 7. USD VE EUR KAYITLARINI ÇIKARMA
# ============================================================

df_core["doviz_cinsi"] = (
    df_core["doviz_cinsi"]
    .astype("string")
    .str.strip()
    .str.upper()
)

foreign_currency_mask = df_core["doviz_cinsi"].isin(
    ["USD", "EUR"]
)

excluded_foreign_currency_count = int(
    foreign_currency_mask.sum()
)

df = df_core.loc[
    ~foreign_currency_mask
].copy()


# Sadece TL bekleniyor
if not df["doviz_cinsi"].isin(["TL"]).all():
    print("Uyarı: TL dışında başka para birimi bulundu.")


# ============================================================
# 8. STATUS FEATURE'LARI
# ============================================================

df["status_norm"] = normalize_status(
    df["siparis_sureci"]
)

df["is_delivered"] = (
    df["status_norm"]
    .eq("teslim edildi")
    .astype("int8")
)

df["is_cancelled"] = (
    df["status_norm"]
    .eq("iptal edildi")
    .astype("int8")
)

df["is_returned"] = (
    df["status_norm"]
    .eq("iade edildi")
    .astype("int8")
)


# Açıklamalı iade dosyasında kaydı var mı?
df["has_return_request"] = (
    df["return_line_count"]
    .notna()
    .astype("int8")
)


# ============================================================
# 9. SİPARİŞ TARİHİ FEATURE'LARI
# ============================================================

df["order_date"] = df["tarih"].dt.normalize()

df = add_date_features(
    df,
    date_column="order_date",
    prefix="order"
)

df["order_hour"] = df["tarih"].dt.hour.astype("Int64")
df["order_minute"] = df["tarih"].dt.minute.astype("Int64")


# Tarih feature'larının döngüsel versiyonları
df["order_month_sin"] = np.sin(
    2 * np.pi * (df["order_month"] - 1) / 12
)

df["order_month_cos"] = np.cos(
    2 * np.pi * (df["order_month"] - 1) / 12
)

df["order_day_of_week_sin"] = np.sin(
    2 * np.pi * df["order_day_of_week"] / 7
)

df["order_day_of_week_cos"] = np.cos(
    2 * np.pi * df["order_day_of_week"] / 7
)

df["order_hour_sin"] = np.sin(
    2 * np.pi * df["order_hour"] / 24
)

df["order_hour_cos"] = np.cos(
    2 * np.pi * df["order_hour"] / 24
)


# ============================================================
# 10. FATURA TARİHİ FEATURE'LARI
# ============================================================

df["invoice_date"] = df["fatura_tarihi"].dt.normalize()

df["is_invoiced"] = (
    df["invoice_date"]
    .notna()
    .astype("int8")
)

df = add_date_features(
    df,
    date_column="invoice_date",
    prefix="invoice"
)

# Fatura tarihi sipariş tarihinden kaç gün sonra?
df["invoice_lag_days"] = (
    df["invoice_date"] - df["order_date"]
).dt.days.astype("Int64")


# ============================================================
# 11. İADE TALEP TARİHİ FEATURE'LARI
# ============================================================

df = add_date_features(
    df,
    date_column="return_request_date",
    prefix="return_request"
)

df["return_lag_days"] = (
    df["return_request_date"] - df["order_date"]
).dt.days.astype("Int64")


# ============================================================
# 12. KAMPANYA / HEDİYE ÇEKİ FEATURE'LARI
# ============================================================

df["is_gift_voucher"] = (
    df["hediye_ceki"]
    .notna()
    .astype("int8")
)

df["is_offer"] = (
    df["kampanya"]
    .notna()
    .astype("int8")
)


# ============================================================
# 13. TUTAR FEATURE'LARI
# ============================================================

df["amount_missing"] = (
    df["tutar"]
    .isna()
    .astype("int8")
)

amount = df["tutar"].fillna(0)

df["realized_sales_amount"] = np.where(
    df["is_delivered"].eq(1),
    amount,
    0
)

df["cancelled_order_amount"] = np.where(
    df["is_cancelled"].eq(1),
    amount,
    0
)

# Bu gerçek para iadesi değil, iade edilen siparişin toplam tutarıdır.
df["returned_order_amount"] = np.where(
    df["is_returned"].eq(1),
    amount,
    0
)


# ============================================================
# 14. GEREKSİZ / KİŞİSEL KOLONLARI ÇIKARMA
# ============================================================

drop_columns = [
    "id",

    # Üye ve kişisel bilgiler
    "uye_adi",
    "firma_uye",
    "cep_telefonu_uye",
    "uye_grup_kodu",
    "uye_grubu",
    "uye_ws_kodu",
    "firma_uye_adi_fatura",
    "e_posta_adresi",
    "vergi_tc_no",
    "vergi_dairesi",
    "vergi_dairesi",

    # Teslimat kişisel bilgileri
    "semt_teslimat",
    "posta_kodu_teslimat",
    "cep_telefonu_teslimat",
    "adres_teslimat",
    "ad_teslimat",

    # Fatura kişisel bilgileri
    "firma_fatura",
    "ad_fatura",
    "semt_fatura",
    "cep_telefonu_fatura",
    "adres_fatura",

    # Takip ve teknik ID alanları
    "kargo_takip_no",
    "kargo_no",
    "kargo_no_iade",
    "fatura_numarasi",
    "irsaliye_numarasi",
    "platform_siparis_no",

    # Gereksiz / yüksek riskli alanlar
    "genel_siparis_notu",
    "alt_odeme_tipi",

    # USD/EUR çıkarıldığı için artık gereksiz
    "doviz_tutar",
    "sistem_kuru",
    "kur_fiyati"
]

df_final = df.drop(
    columns=[
        column for column in drop_columns
        if column in df.columns
    ]
).copy()


# ============================================================
# 15. MODELDE LEAKAGE OLUŞTURMAYACAK FEATURE DOSYASI
# ============================================================

# Bunlar sipariş anında bilinebilecek alanlardır.
# Status, fatura ve iade sonrası oluşan alanlar burada yoktur.

safe_model_columns = [
    "siparis_no",
    "tarih",
    "order_date",

    "order_year",
    "order_month",
    "order_day",
    "order_day_of_week",
    "order_week_of_year",
    "order_quarter",
    "order_is_weekend",
    "order_hour",
    "order_minute",

    "order_month_sin",
    "order_month_cos",
    "order_day_of_week_sin",
    "order_day_of_week_cos",
    "order_hour_sin",
    "order_hour_cos",

    "tutar",
    "kdv",
    "kargo_toplami",
    "hizmet_bedeli",
    "kargo",
    "amount_missing",

    "doviz_cinsi",
    "odeme_tipi",
    "banka",
    "kart",
    "pos",

    "platform",
    "kaynak",
    "araci",

    "il_teslimat",
    "ilce_teslimat",
    "ulke_teslimat",

    "hediye_ceki",
    "kampanya",
    "is_gift_voucher",
    "is_offer"
]

safe_model_columns = [
    column for column in safe_model_columns
    if column in df_final.columns
]

df_model_features = df_final[
    safe_model_columns
].copy()


# ============================================================
# 16. KALİTE RAPORU
# ============================================================

quality_report = pd.DataFrame({
    "metric": [
        "Teslim kayıt sayısı",
        "İptal kayıt sayısı",
        "İade kayıt sayısı",
        "Core toplam kayıt sayısı",
        "Core benzersiz sipariş sayısı",
        "USD/EUR çıkarılan kayıt sayısı",
        "Final TL kayıt sayısı",
        "Açıklamalı iade benzersiz sipariş sayısı",
        "Ana veride bulunmayan iade siparişi sayısı",
        "Eksik fatura tarihi sayısı",
        "Eksik tutar sayısı",
        "İade talebi olan sipariş sayısı",
        "Gerçekleşmiş iade siparişi sayısı",
        "İptal siparişi sayısı",
        "Teslim edilmiş sipariş sayısı"
    ],
    "value": [
        len(df_teslim),
        len(df_iptal),
        len(df_iade),
        len(df_core),
        df_core["siparis_no"].nunique(),
        excluded_foreign_currency_count,
        len(df_final),
        df_iade_aciklamali["siparis_no"].nunique(),
        unmatched_return_detail["siparis_no"].nunique(),
        int(df_final["fatura_tarihi"].isna().sum()),
        int(df_final["tutar"].isna().sum()),
        int(df_final["has_return_request"].sum()),
        int(df_final["is_returned"].sum()),
        int(df_final["is_cancelled"].sum()),
        int(df_final["is_delivered"].sum())
    ]
})


# ============================================================
# 17. DOSYALARI KAYDETME
# ============================================================

merge_output_path = PROCESSED_DIR / "merge_data.xlsx"

with pd.ExcelWriter(
    merge_output_path,
    engine="openpyxl"
) as writer:
    df_final.to_excel(
        writer,
        sheet_name="orders",
        index=False
    )

    quality_report.to_excel(
        writer,
        sheet_name="quality_report",
        index=False
    )


model_output_path = PROCESSED_DIR / "model_features_pre_order.xlsx"

df_model_features.to_excel(
    model_output_path,
    index=False
)


# Ana dosyada bulunmayan iade kayıtlarını ayrıca kaydet
if not unmatched_return_detail.empty:
    unmatched_columns = [
        "siparis_no",
        "talep_tarihi",
        "urun_adi",
        "alt_urun_adi",
        "iade_nedeni",
        "siparis_edilen_miktar",
        "iade_edilecek_miktar"
    ]

    unmatched_columns = [
        column for column in unmatched_columns
        if column in unmatched_return_detail.columns
    ]

    unmatched_return_detail[
        unmatched_columns
    ].to_excel(
        PROCESSED_DIR / "unmatched_return_requests.xlsx",
        index=False
    )


# ============================================================
# 18. KONTROL ÇIKTILARI
# ============================================================

print("İşlem tamamlandı.")
print(f"Final dosya: {merge_output_path}")
print(f"Model feature dosyası: {model_output_path}")
print()
print("Final veri boyutu:", df_final.shape)
print()
print("Status dağılımı:")
print(df_final["status_norm"].value_counts(dropna=False))
print()
print("Para birimi dağılımı:")
print(df_final["doviz_cinsi"].value_counts(dropna=False))
print()
print("USD/EUR olarak çıkarılan kayıt:", excluded_foreign_currency_count)
print("Ana veride bulunmayan iade siparişi:",
      unmatched_return_detail["siparis_no"].nunique())

İşlem tamamlandı.
Final dosya: c:\Users\gizem\DOGO_StajVerileri\dogo-satis-analizi\data\processed\merge_data.xlsx
Model feature dosyası: c:\Users\gizem\DOGO_StajVerileri\dogo-satis-analizi\data\processed\model_features_pre_order.xlsx

Final veri boyutu: (3133, 83)

Status dağılımı:
status_norm
teslim edildi    2655
iptal edildi      249
iade edildi       229
Name: count, dtype: int64[pyarrow]

Para birimi dağılımı:
doviz_cinsi
TL    3133
Name: count, dtype: int64[pyarrow]

USD/EUR olarak çıkarılan kayıt: 41
Ana veride bulunmayan iade siparişi: 30
